# CICE validation figures — PyGMT

Configure real outputs below. The default run creates an **analytical reference only**.
No synthetic figure or successful plot is evidence that CICE passed a gate.
All panels use PyGMT; exports include PNG/PDF and provenance JSON. Install
`CICE_testing[figures]` in a GMT-enabled environment first.

In [ ]:
from pathlib import Path
from CICE_testing import FigureSpec
from CICE_testing.plotting.validation import ValidationFigures

# Edit this one cell. Keep generated output outside accepted run directories.
OUTPUT = Path("validation_report")
SCOPE = "box"                     # "global" for geographical snapshots
RUN = None                        # Path("/path/to/runs/dt_b65_sp_s1")
REFERENCE = None                  # Path("/path/to/matching/reference")
SNAPSHOT = None                   # Path("/path/to/history/iceh.2005-01-01.nc")
RESTART = None                    # Path("/path/to/restart/iced....nc")
GRID_HISTORY = None               # Matching history with tmask/TLAT/TLON/tarea
FIXTURES = {}                     # e.g. {"small": Path("/path/to/small/history/iceh....nc")}
STREAM = "hourly"                 # Keep daily means and hourly instants separate
FIELDS = ("dtens_flarge", "dtens_gcand", "ktens_cand")
EXCLUDE_BLOCK_OWNERSHIP = False    # True only for an intentional decomposition comparison
plots = ValidationFigures(FigureSpec(OUTPUT, scope=SCOPE))

from IPython.display import Image, display
def show_pngs(paths):
    for path in paths:
        if path.suffix == ".png":
            display(Image(filename=str(path)))

In [ ]:
reference_files = plots.mapping()
show_pngs(reference_files)

In [ ]:
if FIXTURES:
    show_pngs(plots.fixture_matrix(FIXTURES))
else:
    print("Configure FIXTURES for measured uniform-fixture points.")

## Measured spatial structure
Box plots use global i/j indices. Global plots use geographic cell centres and
require TLON/TLAT metadata. Select applied fields (`dyntens_g`, `ktens_eff`) or
physical fields separately. A pre-forcing IC wind snapshot is not a forced state.

In [ ]:
if SNAPSHOT is not None:
    snapshot_files = plots.snapshot(SNAPSHOT, names=FIELDS)
    show_pngs(snapshot_files)
else:
    print("Configure SNAPSHOT for measured spatial fields.")

In [ ]:
if RUN is not None:
    for field in ("dtens_gcand", "dyntens_g", "ktens_eff"):
        show_pngs(plots.timeseries(RUN, name=field, stream=STREAM))
else:
    print("Configure RUN for measured temporal evidence.")

## Equivalence evidence
The exported CSV records all variables, masks and excluded ownership fields.
Error figures are descriptive: assess full checker transcripts and gate JSON
before claiming exact restart/decomposition equivalence. Candidate diagnostic
runs and non-diagnostic controls have different inventories; use the B6.5
neutrality validator for that comparison, not this equal-inventory plot.

In [ ]:
if RUN is not None and REFERENCE is not None:
    for folder in ("history", "restart"):
        show_pngs(plots.comparison(RUN, REFERENCE, folder=folder,
                                exclude_block_ownership=EXCLUDE_BLOCK_OWNERSHIP))
else:
    print("Configure matching RUN and REFERENCE for differences.")

In [ ]:
if RESTART is not None and GRID_HISTORY is not None:
    show_pngs(plots.fsd_classes(RESTART, GRID_HISTORY))
else:
    print("Configure RESTART and its matching GRID_HISTORY for FSD diagnosis.")

## Report handoff
After B6.6 completes, select measured figures for
`box_tensile_dev_procedure_results_validation_notes.md`, retaining their manifests,
CSV tables and stage verdicts. Use a separate global record with `SCOPE="global"`
and actual global tests. See `../docs/development_reports.md`.